**JSON — Reading, explode & Nested Data**

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-19")
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-daf782a4-7ca7-41be-b34b-3a7e263dc6a9;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 136ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
# Read a JSON file from S3 — schema inferred automatically, including nested fields
df = spark.read.json("s3a://pyspark-30-days-rahul-2026/data/orders.json")
df.printSchema()
df.show(3, truncate=False)

26/08/09 08:29:22 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- amount: double (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- segment: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- status: string (nullable = true)

+-------+--------------------------------------------+------------------------+----------+--------+---------+
|amount |customer                                    |items                   |order_date|order_id|status   |
+-------+--------------------------------------------+------------------------+----------+--------+---------+
|1299.99|{New York, C001, James Anderson, Enterprise}|[P001]                  |2023-01-05|O0001   |Delivered|
|449.99 |{Los Angeles, C002, Maria Garcia, SMB}      |[P005]                  |2023-01-07|O0002   |Delivere

Spark automatically inferred the nested customer struct and the items array directly from the JSON structure — no schema definition needed. Compare this to CSV, where nesting and arrays simply do not exist as concepts.

In [3]:
# Multi-line JSON (pretty-printed files)
orders_pretty = spark.read.option("multiLine", True).\
    json("/Users/rahulsinghrana/AWS Data Engineering/pyspark 30 days/Data/orders.pretty.json")
orders_pretty.show(3,truncate=False)

+---------------------------------------+-----------------------------------------------------------+----------+--------+---------------------------+-----------------------------+
|customer                               |items                                                      |order_date|order_id|payment                    |shipping_address             |
+---------------------------------------+-----------------------------------------------------------+----------+--------+---------------------------+-----------------------------+
|{C001, rahul@example.com, Rahul Singh} |[{P101, Laptop, 1, 65000}, {P102, Wireless Mouse, 2, 1200}]|2026-08-01|ORD001  |{Credit Card, Success}     |{Noida, India, Uttar Pradesh}|
|{C002, priya@example.com, Priya Sharma}|[{P103, Keyboard, 1, 2500}]                                |2026-08-02|ORD002  |{UPI, Success}             |{Bengaluru, India, Karnataka}|
|{C003, amit@example.com, Amit Kumar}   |[{P104, Monitor, 2, 15000}, {P105, HDMI Cable, 2, 800}]    

In [4]:
#Accessing Nested Fields — StructType
# Access nested fields directly using dot notation
from pyspark.sql import functions as F
df.select(
    F.col("order_id"),
    F.col("customer.id").alias("customer_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.city").alias("city"),
    F.col("customer.segment").alias("segment"),
    F.col("amount")
).show(5, truncate=False)

+--------+-----------+--------------+-----------+----------+-------+
|order_id|customer_id|customer_name |city       |segment   |amount |
+--------+-----------+--------------+-----------+----------+-------+
|O0001   |C001       |James Anderson|New York   |Enterprise|1299.99|
|O0002   |C002       |Maria Garcia  |Los Angeles|SMB       |449.99 |
|O0003   |C003       |Robert Johnson|Chicago    |Enterprise|1399.96|
|O0004   |C004       |Linda Martinez|Houston    |SMB       |179.98 |
|O0005   |C005       |Michael Brown |Phoenix    |Startup   |89.97  |
+--------+-----------+--------------+-----------+----------+-------+
only showing top 5 rows


from_json() — when JSON arrives as a string column

Sometimes JSON arrives nested INSIDE a string column — for example, a CSV column containing a JSON payload, or a Kafka message value. In that case you need from_json() with an explicit schema to parse it into a proper struct.

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# A column that contains a raw JSON string (e.g. from a Kafka topic)
raw_df = spark.createDataFrame(
    [('{"id": "C001", "name": "James Anderson", "city": "New York"}',)],
    ["customer_json"]
)

customer_schema = StructType([
    StructField("id",   StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True)
])

raw_df.withColumn("customer", F.from_json(F.col("customer_json"), customer_schema)) \
      .select("customer.id", "customer.name", "customer.city") \
      .show()

+----+--------------+--------+
|  id|          name|    city|
+----+--------------+--------+
|C001|James Anderson|New York|
+----+--------------+--------+



**Handling Malformed JSON**

In [6]:
# PERMISSIVE is the default — captures bad records in _corrupt_record automatically
# # Rows with a non-null _corrupt_record are the ones Spark could not parse
dirty_json_df = spark.read \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record")  \
    .json("s3a://pyspark-30-days-rahul-2026/data/orders_dirty.json") \
    .cache()

# Materialize the cache
dirty_json_df.count()

dirty_json_df.filter(F.col("_corrupt_record").isNotNull()) \
    .select("_corrupt_record") \
    .show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_corrupt_record                                                                                                                                                                                          |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|THIS IS NOT VALID JSON AT ALL                                                                                                                                                                            |
|{"order_id": "O0005", "customer": {"id": "C005", "name": "Michael Brown", "city": "Phoenix", "segment": "Startup"} "order_date": "2023-01-15", "amount": 89.97, "status": "Delivered", 

Key difference from CSV: For JSON, Spark automatically adds the _corrupt_record column when it encounters structurally broken JSON — you don't need to add it to your schema manually like we did with CSV on Day 3. However, a row with valid JSON syntax but a wrong type (like "amount": "not_a_number") is NOT flagged as corrupt — that field is simply set to null, same behavior as CSV.

**explode() — Flattening Arrays**

JSON often contains arrays — a list of items inside a single field. explode() creates one row per element in the array. This is one of the most important functions for working with JSON data.

In [7]:
# orders.json has an "items" array — one or more product_ids per order
df.select("order_id", "items").show(5, truncate=False)

# explode — one row per item
df.withColumn("product_id", F.explode(F.col("items"))) \
    .select("order_id", "product_id") \
    .show(10)

+--------+------------------------+
|order_id|items                   |
+--------+------------------------+
|O0001   |[P001]                  |
|O0002   |[P005]                  |
|O0003   |[P003, P003, P003, P003]|
|O0004   |[P006, P006]            |
|O0005   |[P002, P002, P002]      |
+--------+------------------------+
only showing top 5 rows
+--------+----------+
|order_id|product_id|
+--------+----------+
|   O0001|      P001|
|   O0002|      P005|
|   O0003|      P003|
|   O0003|      P003|
|   O0003|      P003|
|   O0003|      P003|
|   O0004|      P006|
|   O0004|      P006|
|   O0005|      P002|
|   O0005|      P002|
+--------+----------+
only showing top 10 rows


**explode() on inline data — a second example**

In [8]:
# Customer with multiple orders as an array
data = [
    ("C001", "James", ["O0001", "O0021", "O0046"]),
    ("C002", "Maria", ["O0002", "O0022", "O0047"]),
    ("C003", "Robert", ["O0003"])
]

df = spark.createDataFrame(data, ["customer_id", "name", "order_ids"])
print("Before explode:")
df.show()

# explode — one row per order
print("After explode:")
df.withColumn("order_id", F.explode(F.col("order_ids"))) \
  .select("customer_id", "name", "order_id") \
  .show()

Before explode:
+-----------+------+--------------------+
|customer_id|  name|           order_ids|
+-----------+------+--------------------+
|       C001| James|[O0001, O0021, O0...|
|       C002| Maria|[O0002, O0022, O0...|
|       C003|Robert|             [O0003]|
+-----------+------+--------------------+

After explode:
+-----------+------+--------+
|customer_id|  name|order_id|
+-----------+------+--------+
|       C001| James|   O0001|
|       C001| James|   O0021|
|       C001| James|   O0046|
|       C002| Maria|   O0002|
|       C002| Maria|   O0022|
|       C002| Maria|   O0047|
|       C003|Robert|   O0003|
+-----------+------+--------+



**explode_outer() — keep rows with empty arrays**

In [9]:
# Customer with no orders — explode() drops them, explode_outer() keeps them as null
data = [
    ("C001", "James",  ["O0001", "O0021"]),
    ("C004", "Linda",  []),           # no orders
    ("C005", "Michael", None)         # null array
]

df = spark.createDataFrame(data, ["customer_id", "name", "order_ids"])

print("explode() — drops empty/null arrays:")
df.withColumn("order_id", F.explode(F.col("order_ids"))).show()

print("explode_outer() — keeps empty/null arrays as null:")
df.withColumn("order_id", F.explode_outer(F.col("order_ids"))).show()

explode() — drops empty/null arrays:
+-----------+-----+--------------+--------+
|customer_id| name|     order_ids|order_id|
+-----------+-----+--------------+--------+
|       C001|James|[O0001, O0021]|   O0001|
|       C001|James|[O0001, O0021]|   O0021|
+-----------+-----+--------------+--------+

explode_outer() — keeps empty/null arrays as null:
+-----------+-------+--------------+--------+
|customer_id|   name|     order_ids|order_id|
+-----------+-------+--------------+--------+
|       C001|  James|[O0001, O0021]|   O0001|
|       C001|  James|[O0001, O0021]|   O0021|
|       C004|  Linda|            []|    NULL|
|       C005|Michael|          NULL|    NULL|
+-----------+-------+--------------+--------+



**collect_list() and collect_set() — The Reverse of explode()**

If explode() turns one row with an array into many rows, collect_list() and collect_set() do the reverse — they aggregate multiple rows back into an array.

In [10]:
# Group all order IDs per customer into an array
orders_df=spark.read.csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv',header=True,
inferSchema=True
)
orders_df.groupBy("customer_id").agg(
    F.collect_list("order_id").alias("all_orders"),       # keeps duplicates
    F.collect_set("status").alias("unique_statuses"),     # removes duplicates
    F.count("order_id").alias("order_count")
).orderBy("customer_id").show(5, truncate=False)

+-----------+-----------------------------------+-----------------------+-----------+
|customer_id|all_orders                         |unique_statuses        |order_count|
+-----------+-----------------------------------+-----------------------+-----------+
|C001       |[O0001, O0021, O0046, O0071, O0096]|[Shipped, Delivered]   |5          |
|C002       |[O0002, O0022, O0047, O0072, O0097]|[Shipped, Delivered]   |5          |
|C003       |[O0003, O0023, O0048, O0073, O0098]|[Delivered, Processing]|5          |
|C004       |[O0004, O0024, O0049, O0074, O0099]|[Delivered, Processing]|5          |
|C005       |[O0005, O0025, O0050, O0075, O0100]|[Delivered, Processing]|5          |
+-----------+-----------------------------------+-----------------------+-----------+
only showing top 5 rows


**Task 1**

Create a DataFrame from JSON strings where each row has a nested address object with street, city, and zip. Use from_json() and dot notation to extract each field into its own column.

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

data = [
    ('{"street":"MG Road","city":"Bangalore","zip":"560001"}',),
    ('{"street":"Park Street","city":"Kolkata","zip":"700016"}',),
    ('{"street":"Connaught Place","city":"Delhi","zip":"110001"}',)
]
schema=StructType([
    StructField("street",StringType(),True),
    StructField("city",StringType(),True),
    StructField("zip",StringType(),True)
])
df = spark.createDataFrame(data, ["address_json"])
df.withColumn("Address",F.from_json(F.col("address_json"),schema)
).select("Address.*").show(truncate=False)



+---------------+---------+------+
|street         |city     |zip   |
+---------------+---------+------+
|MG Road        |Bangalore|560001|
|Park Street    |Kolkata  |700016|
|Connaught Place|Delhi    |110001|
+---------------+---------+------+



**Task 2**

Create a DataFrame where each customer has an array of product categories they've purchased. Use explode() to create one row per category. How many total rows does the exploded DataFrame have?

In [12]:
data=[("C001",["Electronics","Books"]),("C002",["Clothing"]),("C003",[]),("C004",None),
      ]
df=spark.createDataFrame(data,["customer_id","categories"])
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("categories", ArrayType(StringType()), True)])
df=df.withColumn('customer',
              F.explode(F.col("categories"))).select("customer_id","customer")
df.show(5,truncate=False)
df.count()


+-----------+-----------+
|customer_id|customer   |
+-----------+-----------+
|C001       |Electronics|
|C001       |Books      |
|C002       |Clothing   |
+-----------+-----------+



3

**Task 3**

Using orders.csv, use collect_list() to group all order_id values per customer_id. Then use collect_set() to get the unique statuses per customer. Show both alongside the order count.

In [13]:
orders_df.groupBy("customer_id").agg(
    F.collect_list("order_id").alias("all_orders"),
    F.collect_set("status").alias("unique_statuses"),
    F.count("*").alias("order_count")
).show(truncate=False)

+-----------+-----------------------------------+-----------------------+-----------+
|customer_id|all_orders                         |unique_statuses        |order_count|
+-----------+-----------------------------------+-----------------------+-----------+
|C006       |[O0006, O0026, O0051, O0076]       |[Delivered, Processing]|4          |
|C010       |[O0010, O0030, O0055, O0080]       |[Delivered, Processing]|4          |
|C007       |[O0007, O0027, O0052, O0077]       |[Delivered, Processing]|4          |
|C018       |[O0018, O0038, O0063, O0088]       |[Shipped, Delivered]   |4          |
|C025       |[O0045, O0070, O0095]              |[Shipped, Delivered]   |3          |
|C012       |[O0012, O0032, O0057, O0082]       |[Delivered, Processing]|4          |
|C003       |[O0003, O0023, O0048, O0073, O0098]|[Delivered, Processing]|5          |
|C023       |[O0043, O0068, O0093]              |[Shipped, Delivered]   |3          |
|C015       |[O0015, O0035, O0060, O0085]       |[Canc

**Task 4**


Create a DataFrame with a mix of customers — some with order arrays, some with empty arrays, some with null. Use both explode() and explode_outer() and compare the row counts. Which one should you use in production and why?

In [14]:

data = [
    ("C101", ["O1", "O2", "O3"]),
    ("C102", []),
    ("C103", None),
    ("C104", ["O4"])
]

customers_df = spark.createDataFrame(data, ["customer_id", "orders"])

customers_df.show(truncate=False)
explode_df = customers_df.select(
    "customer_id",
    F.explode("orders").alias("order_id")
)

explode_df.show()
explode_outer_df = customers_df.select(
    "customer_id",
    F.explode_outer("orders").alias("order_id")
)

explode_outer_df.show()
explode_outer_df.count()

+-----------+------------+
|customer_id|orders      |
+-----------+------------+
|C101       |[O1, O2, O3]|
|C102       |[]          |
|C103       |NULL        |
|C104       |[O4]        |
+-----------+------------+

+-----------+--------+
|customer_id|order_id|
+-----------+--------+
|       C101|      O1|
|       C101|      O2|
|       C101|      O3|
|       C104|      O4|
+-----------+--------+

+-----------+--------+
|customer_id|order_id|
+-----------+--------+
|       C101|      O1|
|       C101|      O2|
|       C101|      O3|
|       C102|    NULL|
|       C103|    NULL|
|       C104|      O4|
+-----------+--------+



6

In [15]:
spark.stop()